<a href="https://colab.research.google.com/github/ElCeasaR7/cloud-automation/blob/main/%D9%85%D8%B4%D8%B1%D9%88%D8%B9%20%D8%B1%D8%A7%D8%AC%20%D8%B9%D9%84%D9%89%20%D8%A7%D8%AC%D9%8A%D9%86%D8%AC%20%D9%85%D9%88%D8%B1%D8%AF%D9%8A%D9%86%205%20%D9%8A%D9%88%D9%85%2024-5-2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# تثبيت المكتبات بنسخ مستقرة ومتوافقة مع دعم الإكسيل
!pip install -q \
langchain \
langchain-community \
langchain-text-splitters \
sentence-transformers \
chromadb \
pypdf \
python-docx \
flask \
pyngrok \
requests \
openpyxl \
pandas

print("✅ تم تثبيت كل المكتبات بنجاح!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.8/343.8 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/1

In [2]:
from google.colab import files
uploaded = files.upload()

Saving Supplier Aging          05-2026.xlsx to Supplier Aging          05-2026.xlsx


In [3]:
import os
import shutil
import pandas as pd
from pypdf import PdfReader
from docx import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# --- 1. قراءة ملف الإكسيل وتحويله لنص ---
all_text = ""
for file_name in uploaded.keys():
    print(f"🔄 جاري قراءة المستند: {file_name}")
    if file_name.endswith(".xlsx") or file_name.endswith(".xls"):
        excel_file = pd.read_excel(file_name, sheet_name=None)
        for sheet_name, df in excel_file.items():
            all_text += f"\n--- اسم ورقة العمل (Sheet): {sheet_name} ---\n"
            all_text += df.to_string(index=False) + "\n"
    elif file_name.endswith(".pdf"):
        reader = PdfReader(file_name)
        for page in reader.pages:
            text = page.extract_text()
            if text: all_text += text + "\n"
    elif file_name.endswith(".txt"):
        with open(file_name, "r", encoding="utf-8") as f: all_text += f.read() + "\n"
    elif file_name.endswith(".docx"):
        doc = Document(file_name)
        for para in doc.paragraphs: all_text += para.text + "\n"

print(f"📊 إجمالي الحروف المقروءة: {len(all_text)}")

# --- 2. تقسيم النصوص (Chunking) لحفظ جداول الحسابات ---
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", " ", ""]
)
chunks = splitter.split_text(all_text)
print(f"📦 إجمالي عدد القطع (Chunks): {len(chunks)}")

# --- 3. تحميل موديل الـ Embeddings الداعم للعربية ---
print("🔄 جاري تحميل موديل الـ Embeddings العربي (LaBSE)...")
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/LaBSE")

# --- 4. إنشاء الـ Vector DB (Chroma) ---
persist_directory = "rag_db"
if os.path.exists(persist_directory):
    shutil.rmtree(persist_directory)

print("⏳ جاري تحويل البيانات وتخزينها في Chroma... (سيبها تخلص براحتها)")
vector_db = Chroma.from_texts(
    texts=chunks,
    embedding=embedding_model,
    persist_directory=persist_directory
)

print("🎯 مبروك! قاعدة البيانات جاهزة ومستقرة تماماً الآن!")

/tmp/ipykernel_6642/2825277520.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


🔄 جاري قراءة المستند: Supplier Aging          05-2026.xlsx
📊 إجمالي الحروف المقروءة: 298542
📦 إجمالي عدد القطع (Chunks): 536
🔄 جاري تحميل موديل الـ Embeddings العربي (LaBSE)...


/tmp/ipykernel_6642/2825277520.py:43: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/LaBSE")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

⏳ جاري تحويل البيانات وتخزينها في Chroma... (سيبها تخلص براحتها)
🎯 مبروك! قاعدة البيانات جاهزة ومستقرة تماماً الآن!


In [4]:
# اكتب هنا السؤال اللي عايز تبحث بيه في ملف أعمار الموردين
# مثال: اكتب اسم مورد معين موجود في الملف عندك، أو كلمة "مستحق"
query = "اكتب اسم مورد من الملف هنا أو اكتب: ما هي المبالغ المستحقة؟"

# البحث عن أفضل 3 قطع متعلقة بالسؤال
results = vector_db.similarity_search(query, k=3)

print(f"🔍 نتائج البحث عن: '{query}'\n")

for i, result in enumerate(results):
    print("="*60)
    print(f"📄 القطعة المسترجعة رقم {i+1}:")
    print("="*60)
    print(result.page_content)
    print("\n")

🔍 نتائج البحث عن: 'اكتب اسم مورد من الملف هنا أو اكتب: ما هي المبالغ المستحقة؟'

📄 القطعة المسترجعة رقم 1:
--- اسم ورقة العمل (Sheet): Supplier Aging 05-2026.xlsx - L ---
رقم الفاتورة تاريخ الفاتورة  رقم إذن الاستلام  رقم أمر الشراء  رقم طلب الشراء                اسم الصنف                               اسم المورد  أصل المبلغ      0.14     0.01  0.03  صافي المبلغ               ملاحظات الرقم الضريبى  Unnamed: 14  Unnamed: 15  Unnamed: 16      0.14.1  0.01.1
        1693     2026-05-10            8019.0          4365.0          8302.0                      بيض                            اولاد الشرقية    10800.00    0.0000 108.0000   0.0   10692.0000                    تم     302908676          NaN          NaN          NaN         NaN     NaN


📄 القطعة المسترجعة رقم 2:
--- اسم ورقة العمل (Sheet): Sheet1 ---


📄 القطعة المسترجعة رقم 3:
NaN         NaN        NaN         NaN                                     اسم المستفيد                               اسم الشركة       Total 2026-04-01 00:0

In [5]:
import google.generativeai as genai
from google.colab import userdata

# سحب المفتاح بأمان من الـ Secrets بتاعتك
try:
    GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
    # استخدام الموديل الأحدث والأسرع
    model = genai.GenerativeModel('gemini-1.5-flash')
    print("✅ تم استدعاء الـ API Key بنجاح والاتصال بـ Gemini جاهز!")
except Exception as e:
    print(f"❌ حدث خطأ في سحب المفتاح، تأكد من الاسم: {str(e)}")

# دالة الـ RAG الكاملة المستقرة على Colab
def rag_search_on_colab(question, k=3):
    # 1. البحث بالمعنى في الـ Vector DB اللي عملناه للإكسيل
    docs = vector_db.similarity_search(question, k=k)
    context = "\n\n".join([doc.page_content for doc in docs])

    # 2. صياغة البرومبت المحاسبي الذكي
    prompt = f"""
أنت مساعد ذكي ومحاسب مالي محترف.
بناءً على البيانات المفسرة والمستخرجة التالية من نظام أعمار الموردين (Supplier Aging)، أجب على سؤال المستخدم بشكل دقيق ومنظم ومنسق جداً (استخدم جداول Markdown أو نقاط واضحة).
ملاحظة: العملة الأساسية هي الجنيه المصري (EGP).

البيانات المستخرجة من الإكسيل:
{context}

سؤال المستخدم:
{question}

الإجابة المنسقة للمحاسب:
"""

    # 3. توليد الإجابة من Gemini
    response = model.generate_content(prompt)
    return response.text

print("🚀 دالة البحث الـ RAG جاهزة الآن للاستخدام!")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ تم استدعاء الـ API Key بنجاح والاتصال بـ Gemini جاهز!
🚀 دالة البحث الـ RAG جاهزة الآن للاستخدام!


In [6]:
# اكتب هنا سؤالك المالي
# مثال: "ما هي تفاصيل فاتورة أولاد الشرقية؟" أو "ملخص سريع لبيانات الموردين"
question = "ما هي تفاصيل فاتورة أولاد الشرقية وما هو صافي المبلغ؟"

# تشغيل الـ RAG
print("⏳ جاري البحث وتحليل البيانات الحسابية...")
answer = rag_search_on_colab(question)

print("\n" + "="*50)
print("🤖 إجابة المساعد الذكي المنسقة:")
print("="*50)
print(answer)

⏳ جاري البحث وتحليل البيانات الحسابية...


NotFound: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.

In [8]:
import google.generativeai as genai
from google.colab import userdata

try:
    GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)

    # 🟢 التعديل هنا: استخدام gemini-pro المتوافق تماماً مع المكتبة الحالية
    model = genai.GenerativeModel('gemini-pro')
    print("✅ تم تحديث الموديل إلى gemini-pro والاتصال جاهز!")
except Exception as e:
    print(f"❌ خطأ: {str(e)}")

# دالة الـ RAG بعد تعديل الموديل
def rag_search_on_colab(question, k=3):
    docs = vector_db.similarity_search(question, k=k)
    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
أنت مساعد ذكي ومحاسب مالي محترف.
بناءً على البيانات المفسرة والمستخرجة التالية من نظام أعمار الموردين (Supplier Aging)، أجب على سؤال المستخدم بشكل دقيق ومنظم ومنسق جداً (استخدم جداول Markdown أو نقاط واضحة).
ملاحظة: العملة الأساسية هي الجنيه المصري (EGP).

البيانات المستخرجة من الإكسيل:
{context}

سؤال المستخدم:
{question}

الإجابة المنسقة للمحاسب:
"""
    response = model.generate_content(prompt)
    return response.text

print("🚀 الدالة المحدثة جاهزة للاستخدام!")

✅ تم تحديث الموديل إلى gemini-pro والاتصال جاهز!
🚀 الدالة المحدثة جاهزة للاستخدام!


In [9]:
# سؤال الاختبار المالي
question = "ما هي تفاصيل فاتورة أولاد الشرقية وما هو صافي المبلغ؟"

print("⏳ جاري البحث وتحليل البيانات الحسابية عبر Gemini...")

try:
    # استدعاء الدالة المحدثة اللي لقطت الـ Secret خلاص
    answer = rag_search_on_colab(question)

    print("\n" + "="*50)
    print("🤖 إجابة المساعد الذكي المنسقة والمنظمة:")
    print("="*50)
    print(answer)
except Exception as e:
    print(f"❌ حدث خطأ أثناء توليد الإجابة: {str(e)}")

⏳ جاري البحث وتحليل البيانات الحسابية عبر Gemini...


❌ حدث خطأ أثناء توليد الإجابة: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-pro:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-pro is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.


In [11]:
# 1. تثبيت المكتبة الحديثة الرسمية من جوجل
!pip install -q google-genai

import os
from google.colab import userdata
from google.genai import Client

print("🔄 جاري الاتصال بـ Gemini عبر المكتبة الجديدة...")

try:
    # سحب المفتاح اللي أنت مغير اسمه والـ Colab لقطه
    api_key = userdata.get('GEMINI_API_KEY')

    # تعريف العميل (Client) بالطريقة الحديثة المستقرة
    client = Client(api_key=api_key)
    print("✅ تم الاتصال بنجاح باستخدام المكتبة الجديدة!")
except Exception as e:
    print(f"❌ خطأ في سحب المفتاح: {str(e)}")

# 2. دالة الـ RAG الحديثة
def rag_search_modern(question, k=3):
    # البحث في قاعدة البيانات اللي عملناها للإكسيل
    docs = vector_db.similarity_search(question, k=k)
    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
أنت مساعد ذكي ومحاسب مالي محترف.
بناءً على البيانات المفسرة والمستخرجة التالية من نظام أعمار الموردين (Supplier Aging)، أجب على سؤال المستخدم بشكل دقيق ومنظم ومنسق جداً (استخدم جداول Markdown أو نقاط واضحة).
ملاحظة: العملة الأساسية هي الجنيه المصري (EGP).

البيانات المستخرجة من الإكسيل:
{context}

سؤال المستخدم:
{question}

الإجابة المنسقة للمحاسب:
"""

    # استدعاء الموديل الحديث المستقر بالكامل حالياً
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
    )
    return response.text

# --- التشغيل والاختبار فوراً ---
question = "ما هي تفاصيل فاتورة أولاد الشرقية وما هو صافي المبلغ؟"
print("⏳ جاري البحث والتحليل وتنسيق الجدول المالي الآن...")

try:
    answer = rag_search_modern(question)
    print("\n" + "="*50)
    print("🤖 إجابة المساعد الذكي المنسقة (النسخة المستقرة):")
    print("="*50)
    print(answer)
except Exception as e:
    print(f"❌ حدث خطأ: {str(e)}")

🔄 جاري الاتصال بـ Gemini عبر المكتبة الجديدة...
✅ تم الاتصال بنجاح باستخدام المكتبة الجديدة!
⏳ جاري البحث والتحليل وتنسيق الجدول المالي الآن...

🤖 إجابة المساعد الذكي المنسقة (النسخة المستقرة):
أهلاً بك،

بناءً على البيانات المستخرجة من نظام أعمار الموردين (Supplier Aging 05-2026)، إليك تفاصيل فاتورة المورد "اولاد الشرقية" المطلوبة:

### تفاصيل فاتورة المورد "اولاد الشرقية"

| التفصيل             | القيمة                 |
| :------------------- | :--------------------- |
| **اسم المورد**       | اولاد الشرقية          |
| **رقم الفاتورة**     | 1693                   |
| **تاريخ الفاتورة**   | 2026-05-10             |
| **رقم إذن الاستلام** | 8019                   |
| **رقم أمر الشراء**   | 4365                   |
| **رقم طلب الشراء**   | 8302                   |
| **اسم الصنف**        | بيض                    |
| **أصل المبلغ**       | 10,800.00 جنيه مصري    |
| **ملاحظات**          | تم                     |
| **الرقم الضريبي**    | 302908676              |

### صافي المبلغ

صافي 

In [12]:
import os

# إنشاء الفولدر المخصص لصفحات Flask
os.makedirs('templates', exist_ok=True)

# كتابة كود الـ HTML والتصميم جوه الفولدر
html_content = """
<!DOCTYPE html>
<html lang="ar" dir="rtl">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>المساعد المالي الذكي - Supplier Aging</title>
    <link href="https://fonts.googleapis.com/css2?family=Cairo:wght@400;700&display=swap" rel="stylesheet">
    <style>
        body { font-family: 'Cairo', sans-serif; background-color: #f4f6f9; margin: 0; padding: 20px; color: #333; }
        .container { max-width: 800px; margin: 30px auto; background: white; padding: 30px; border-radius: 12px; box-shadow: 0 4px 15px rgba(0,0,0,0.05); }
        h1 { text-align: center; color: #1e3a8a; font-size: 24px; margin-bottom: 25px; }
        .form-group { margin-bottom: 20px; }
        label { display: block; font-weight: bold; margin-bottom: 8px; color: #4b5563; }
        textarea { width: 100%; height: 100px; padding: 12px; border: 1px solid #d1d5db; border-radius: 8px; font-size: 16px; box-sizing: border-box; resize: vertical; font-family: inherit; }
        button { background-color: #2563eb; color: white; border: none; padding: 12px 25px; border-radius: 8px; font-size: 16px; cursor: pointer; font-weight: bold; width: 100%; transition: background 0.3s; }
        button:hover { background-color: #1d4ed8; }
        .result-box { margin-top: 30px; padding: 20px; background: #f8fafc; border-right: 5px solid #2563eb; border-radius: 8px; display: none; }
        .result-title { font-weight: bold; color: #1e3a8a; margin-bottom: 10px; font-size: 18px; }
        .result-content { line-height: 1.8; font-size: 15px; white-space: pre-wrap; }
        /* تنسيق الجداول المرجعة من الذكاء الاصطناعي */
        table { width: 100%; border-collapse: collapse; margin-top: 15px; background: white; }
        th, td { border: 1px solid #cbd5e1; padding: 10px; text-align: right; }
        th { background-color: #f1f5f9; color: #1e3a8a; }
    </style>
</head>
<body>

<div class="container">
    <h1>📊 المساعد المحاسبي الذكي (RAG System)</h1>

    <div class="form-group">
        <label for="question">اطرح سؤالك المالي حول أعمار الموردين:</label>
        <textarea id="question" placeholder="مثال: ما هي تفاصيل فاتورة أولاد الشرقية؟"></textarea>
    </div>

    <button id="submit-btn" onclick="sendQuery()">⏳ جاري البحث والتحليل...</button>

    <div id="result-box" class="result-box">
        <div class="result-title">🤖 إجابة المساعد الذكي:</div>
        <div id="result-content" class="result-content"></div>
    </div>
</div>

<script>
    // تغيير نص الزرار للوضع العادي عند فتح الصفحة
    document.getElementById("submit-btn").innerText = "إرسال السؤال المالي";

    function sendQuery() {
        const question = document.getElementById("question").value;
        const btn = document.getElementById("submit-btn");
        const resultBox = document.getElementById("result-box");
        const resultContent = document.getElementById("result-content");

        if (!question.trim()) {
            alert("من فضلك اكتب سؤال أولاً!");
            return;
        }

        // تغيير حالة الزرار أثناء المعالجة
        btn.innerText = "🔄 جاري البحث والتحليل وتنسيق البيانات...";
        btn.disabled = true;

        // إرسال الطلب لسيرفر Flask
        fetch("/search", {
            method: "POST",
            headers: { "Content-Type": "application/x-www-form-urlencoded" },
            body: "question=" + encodeURIComponent(question)
        })
        .then(response => response.json())
        .then(data => {
            // عرض النتيجة
            resultContent.innerHTML = data.answer;
            resultBox.style.display = "block";

            // إعادة الزرار لوضعه الطبيعي
            btn.innerText = "إرسال السؤال المالي";
            btn.disabled = false;
        })
        .catch(error => {
            console.error("Error:", error);
            alert("حدث خطأ في الاتصال بالسيرفر!");
            btn.innerText = "إرسال السؤال المالي";
            btn.disabled = false;
        });
    }
</script>

</body>
</html>
"""

with open('templates/index.html', 'w', encoding='utf-8') as f:
    f.write(html_content)

print("✅ تم إنشاء ملف الـ HTML والتصميم بنجاح جوه فولدر templates!")

✅ تم إنشاء ملف الـ HTML والتصميم بنجاح جوه فولدر templates!


In [13]:
from flask import Flask, render_template, request, jsonify
import threading
import urllib.request

app = Flask(__name__)

# دالة تحويل الـ Markdown (مثل الجداول والنقاط) إلى HTML
def format_markdown_to_html(text):
    lines = text.split('\n')
    html_output = ""
    in_table = False

    for line in lines:
        if '|' in line:
            if not in_table:
                html_output += "<table>"
                in_table = True
            if '---' in line: continue
            cells = [f"<td>{c.strip()}</td>" for c in line.split('|')[1:-1]]
            html_output += f"<tr>{''.join(cells)}</tr>"
        else:
            if in_table:
                html_output += "</table>"
                in_table = False

            if line.startswith('### '):
                html_output += f"<h3>{line[4:]}</h3>"
            elif line.startswith('**') and line.endswith('**'):
                html_output += f"<p><strong>{line[2:-2]}</strong></p>"
            elif line.strip():
                html_output += f"<p>{line}</p>"

    if in_table: html_output += "</table>"
    return html_output

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/search", methods=["POST"])
def search():
    user_question = request.form.get("question")
    if not user_question:
        return jsonify({"answer": "من فضلك اكتب سؤالك أولاً!"})
    try:
        raw_answer = rag_search_modern(user_question)
        formatted_answer = format_markdown_to_html(raw_answer)
        return jsonify({"answer": formatted_answer})
    except Exception as e:
        return jsonify({"answer": f"حدث خطأ أثناء المعالجة: {str(e)}"})

# 1. تشغيل Flask في الخلفية على بورت 5000
def run_flask():
    app.run(port=5000, host="0.0.0.0", use_reloader=False)

threading.Thread(target=run_flask).start()

# 2. جلب رقم الـ IP الخاص بالسيرفر (عشان أداة localtunnel بتطلبه كأمان أول ما تفتح الرابط)
print("\n" + "="*60)
print("🔑 الرقم السري (Password/Endpoint IP) الخاص بك هو:")
print(urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())
print("="*60 + "\n")

# 3. تشغيل localtunnel وفتح الرابط فوراً
!npx localtunnel --port 5000


🔑 الرقم السري (Password/Endpoint IP) الخاص بك هو:
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


34.177.117.71

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇your url is: https://slow-socks-matter.loca.lt


INFO:werkzeug:127.0.0.1 - - [24/May/2026 12:55:06] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/May/2026 12:55:07] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [24/May/2026 12:56:30] "POST /search HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/May/2026 12:57:20] "POST /search HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/May/2026 12:58:24] "POST /search HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/May/2026 12:59:12] "POST /search HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/May/2026 13:05:16] "POST /search HTTP/1.1" 200 -


^C


In [ ]:
y